# NB3 — ICDM Consolidate & Diagnose

**No new compute (spec §6).** Merges the regenerated NB1 (CAPGD protocol grid)
and NB2 (Square model-family) registries into the single-provenance
`icdm_master_registry.csv`, then derives every paper-facing diagnostic:

1. Master registry (§6.1) — 294 + 96 = 390 rows, §3.1 schema.
2. Coverage + ICDM summary tables (§6.2), cells tagged with §2 design axes.
3. Kendall-Tau ranking stability (§6.3): **strong** (defences, B vs C1/C2,
   CAPGD) + **free** (model families, A vs B, Square).
4. PR-AUC vs ROC-AUC (§6.4) — the rare-class-collapse evidence.
5. Thesis-consistency cross-check (§6.5) vs the old `registry_clean.csv`
   (read-only; `*baseline` experiments are the canonical comparison — their
   CCFD 3-seed std reproduces the thesis 0.581±0.102 exactly).
6. Golden-reference anchors (§4.4 / §6.6) — warn-only self-check.
7. `fig_kendall_tau` (+ free variant) bump charts (§6.7).
8. `experiment_status.md` (§8 template, filled with computed findings).

**Runs locally** (inputs are committed in the repo) or anywhere the repo is
checked out — no GPU, no Drive, no datasets needed.

In [ ]:
# Cell 1: Paths + inputs (all repo-local; read-only except results/icdm_2026 outputs)
import os
import numpy as np
import pandas as pd

# Run from the repo root (works when launched from notebooks/ too).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
assert os.path.exists("results/icdm_2026/capgd_grid_results.csv"), os.getcwd()

OUT_DIR = "results/icdm_2026"
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

PROTO_NA = "not_applicable"   # NB2's spec-§5.2 Protocol-C placeholder marker

nb1 = pd.read_csv(os.path.join(OUT_DIR, "capgd_grid_results.csv"))
nb2 = pd.read_csv(os.path.join(OUT_DIR, "square_family_results.csv"))
old = pd.read_csv("results/registry_clean.csv")   # read-only cross-check (spec §6.5)

assert len(nb1) == 294 and nb1["run_id"].is_unique
assert len(nb2) == 96 and nb2["run_id"].is_unique
assert list(nb1.columns) == list(nb2.columns), "NB1/NB2 schema drift"
print(f"NB1 {len(nb1)} rows | NB2 {len(nb2)} rows | old registry {len(old)} rows (read-only)")

In [ ]:
# Cell 2: Master registry — union of NB1 + NB2 (spec §6.1, §3.1 schema)
master = pd.concat([nb1, nb2], ignore_index=True)
assert master["run_id"].is_unique and len(master) == 390
MASTER_CSV = os.path.join(OUT_DIR, "icdm_master_registry.csv")
master.to_csv(MASTER_CSV, index=False)

# measured = everything except NB2's Protocol-C placeholders.
measured = master[master["protocol"] != PROTO_NA]
print(f"Saved {len(master)} rows -> {MASTER_CSV} "
      f"({len(measured)} measured + {(master['protocol'] == PROTO_NA).sum()} placeholders)")

In [ ]:
# Cell 3: Coverage table + ICDM summary table (spec §6.2; §2 design axes)
def axes_for(row):
    """Spec-§2 design axes this coverage cell fills. Placeholder cells fill
    none (deferred future work). Axis 2 = the NB2 Square model-family layer;
    axis 3 = the NB1 CAPGD three-protocol diagnostic."""
    if row["protocol"] == PROTO_NA:
        return ""
    axes = {1, 5}                         # scenario+constraints, reporting lens
    if row["attack"] == "Square":
        axes.add(2)                       # model-family coverage (NB2)
    if row["attack"] == "CAPGD":
        axes.add(3)                       # three-protocol diagnostic (NB1)
    if row["protocol"] == "C2_mask_projection":
        axes.add(4)                       # attacker-capability branch
    if row["defence"] != "none":
        axes.add(6)                       # defence loop
    if row["n_eps"] > 1:
        axes.add(7)                       # perturbation-budget sweep
    return ",".join(str(a) for a in sorted(axes))

cov = (master.groupby(["dataset", "model", "defence", "attack", "protocol"])
       .agg(n_rows=("run_id", "size"), n_seeds=("seed", "nunique"),
            n_eps=("epsilon", "nunique"))
       .reset_index())
cov["status"] = np.where(cov["protocol"] == PROTO_NA, "deferred (Cartella-style future work)",
                np.where(cov["protocol"] == "B_posthoc_filter", "derived from A", "measured"))
cov["design_axes"] = cov.apply(axes_for, axis=1)
COVERAGE_CSV = os.path.join(OUT_DIR, "icdm_coverage_table.csv")
cov.to_csv(COVERAGE_CSV, index=False)
print(f"Saved coverage table ({len(cov)} cells) -> {COVERAGE_CSV}")

# ICDM summary table: §3.3-style pivot over seeds for the unioned measured rows.
metrics_m = ["robust_pr_auc", "robust_roc_auc", "flipped_count",
             "feasible_flipped_count", "fsr", "aggregate_feasibility", "robust_accuracy"]
g = measured.groupby(["dataset", "model", "defence", "attack", "protocol", "epsilon"])
summ = g[metrics_m].agg(["mean", "std"])
summ.columns = [f"{stat}_{m}" for m, stat in summ.columns]
summ = summ.reset_index()
summ["n_seeds"] = g.size().values
SUMMARY_CSV = os.path.join(OUT_DIR, "icdm_summary_table.csv")
summ.to_csv(SUMMARY_CSV, index=False)
print(f"Saved ICDM summary table ({len(summ)} groups) -> {SUMMARY_CSV}")

In [ ]:
# Cell 4: Kendall-Tau protocol-ranking stability (spec §6.3)
from scipy.stats import kendalltau, weightedtau

def rank_str(items, scores):
    """Items ordered best-first (higher robust PR-AUC = more robust)."""
    order = np.argsort(-np.asarray(scores))
    return " > ".join(np.asarray(items)[order])

def tau_row(kind, dataset, left, right, items, s_left, s_right):
    """Compare the ranking induced by `s_left` (protocol `left`) against
    `s_right`. kt_distance = (1 - weighted_tau) / 2 in [0, 1]; 0 = identical.
    weighted_tau uses scipy's rank=True default: importance ranks derive from
    the scores themselves (decreasing — the most-robust item weighted highest,
    hyperbolically) and the statistic is averaged over both orderings, i.e. a
    SYMMETRIC top-weighted comparison; neither protocol is the reference.
    NaN propagates if a score vector is constant (n=3 rankings)."""
    tau = kendalltau(s_left, s_right).statistic
    wtau = weightedtau(s_left, s_right).statistic
    return {
        "comparison": kind, "dataset": dataset,
        "protocol_left": left, "protocol_right": right,
        "items": "|".join(items),
        "ranking_left": rank_str(items, s_left),
        "ranking_right": rank_str(items, s_right),
        "kendall_tau": round(float(tau), 4), "weighted_tau": round(float(wtau), 4),
        "kt_distance": round((1 - float(wtau)) / 2, 4),
    }

rows = []

# Strong (spec §6.3.1): rank the MLP defence configs by CAPGD robust PR-AUC
# under B vs under C (both C1 and C2 reported), per constrained dataset, eps=0.1.
DEFENCES = ["none", "adversarial_training", "input_validation"]
capgd = measured[(measured["attack"] == "CAPGD") & (measured["epsilon"] == 0.1)]
for ds in ["IEEE-CIS", "LCLD", "Sparkov"]:          # CCFD has no Protocol C
    sub = capgd[capgd["dataset"] == ds]
    by = {p: [sub[(sub["defence"] == d) & (sub["protocol"] == p)]["robust_pr_auc"].mean()
              for d in DEFENCES]
          for p in ["B_posthoc_filter", "C1_projection", "C2_mask_projection"]}
    for c_proto in ["C1_projection", "C2_mask_projection"]:
        rows.append(tau_row("strong_defence_B_vs_C", ds, "B_posthoc_filter", c_proto,
                            DEFENCES, by["B_posthoc_filter"], by[c_proto]))

# Free (spec §6.3.2): rank the model families by Square robust PR-AUC, A vs B.
MODELS = ["MLP", "XGBoost", "Ensemble"]
square = measured[measured["attack"] == "Square"]
for ds in ["CCFD", "IEEE-CIS", "LCLD", "Sparkov"]:
    sub = square[square["dataset"] == ds]
    by = {p: [sub[(sub["model"] == m) & (sub["protocol"] == p)]["robust_pr_auc"].mean()
              for m in MODELS]
          for p in ["A_unconstrained", "B_posthoc_filter"]}
    rows.append(tau_row("free_model_A_vs_B", ds, "A_unconstrained", "B_posthoc_filter",
                        MODELS, by["A_unconstrained"], by["B_posthoc_filter"]))

kt = pd.DataFrame(rows)
KT_CSV = os.path.join(OUT_DIR, "kendall_tau_protocol_ranking.csv")
kt.to_csv(KT_CSV, index=False)
print(f"Saved {len(kt)} ranking comparisons -> {KT_CSV}\n")
print(kt[["comparison", "dataset", "protocol_right", "ranking_left", "ranking_right",
          "weighted_tau", "kt_distance"]].to_string(index=False))

In [ ]:
# Cell 5: PR-AUC vs ROC-AUC (spec §6.4) — justifies PR-AUC over the ROC-AUC
# convention of prior fraud benchmarks. No-defence MLP, Protocol A, the FULL
# eps sweep + both attacks. Three findings, all verified against the committed
# data before this cell was written:
#   (a) clean ROC-AUC overstates performance on every dataset (Sparkov 0.960
#       vs clean PR-AUC 0.626) — the classic rare-class gap;
#   (b) at small budgets ROC-AUC can stay high while PR-AUC has already
#       collapsed (Sparkov CAPGD eps=0.01: ROC 0.821 vs PR 0.148);
#   (c) at large budgets ROC-AUC inverts BELOW chance (LCLD eps>=0.05 -> 0.0),
#       a pathology PR-AUC does not exhibit (its floor is the positive rate).
nodef = measured[(measured["defence"] == "none") & (measured["model"] == "MLP")
                 & (measured["protocol"] == "A_unconstrained")]
pv = (nodef.groupby(["dataset", "attack", "epsilon"])
      [["clean_pr_auc", "robust_pr_auc", "clean_roc_auc", "robust_roc_auc"]]
      .mean().reset_index())
pv["clean_gap_roc_minus_pr"] = pv["clean_roc_auc"] - pv["clean_pr_auc"]
pv["pr_drop"] = pv["clean_pr_auc"] - pv["robust_pr_auc"]
pv["roc_drop"] = pv["clean_roc_auc"] - pv["robust_roc_auc"]
# (b): robust ROC still "healthy" while robust PR has collapsed.
pv["rocauc_hides_collapse"] = (pv["robust_roc_auc"] > 0.7) & (pv["robust_pr_auc"] < 0.2)
# (c): ranking inversion — ROC below chance under attack.
pv["rocauc_inverted"] = pv["robust_roc_auc"] < 0.5
assert pv["rocauc_hides_collapse"].any(), "expected >=1 hide-collapse row (Sparkov eps=0.01)"
assert pv["rocauc_inverted"].any(), "expected >=1 ROC-inversion row (LCLD large eps)"
PRROC_CSV = os.path.join(OUT_DIR, "prauc_vs_rocauc.csv")
pv.round(4).to_csv(PRROC_CSV, index=False)
print(f"Saved {len(pv)} rows -> {PRROC_CSV}\n")
print(pv.round(3).to_string(index=False))

In [ ]:
# Cell 6: Thesis-consistency cross-check vs old registry_clean.csv (spec §6.5)
# Canonical old rows = the *baseline experiments: their CCFD 3-seed mean/std
# reproduce the thesis 0.581±0.102 exactly. The *eps_sweep duplicates at
# eps=0.1 are excluded (same config re-run; including them would understate
# the variance the spec asks us to reconcile).
NAME_MAP = {"ccfd": "CCFD", "ieee_cis": "IEEE-CIS", "lcld": "LCLD", "sparkov": "Sparkov"}
old_a = old[(old["attack_type"] == "capgd") & (old["model_type"] == "neural")
            & (old["defence_type"] == "none") & (old["attack_epsilon"] == 0.1)
            & (old["experiment_name"].str.endswith("baseline"))].copy()
old_a["dataset"] = old_a["dataset"].map(NAME_MAP)
assert old_a["dataset"].notna().all() and (old_a.groupby("dataset").size() == 3).all(), \
    "expected exactly 3 *baseline rows per dataset in the old registry"
old_g = old_a.groupby("dataset")["robust_pr_auc"].agg(old_mean="mean", old_std="std", old_n="count")

fresh_a = measured[(measured["attack"] == "CAPGD") & (measured["defence"] == "none")
                   & (measured["epsilon"] == 0.1) & (measured["protocol"] == "A_unconstrained")]
fresh_g = fresh_a.groupby("dataset")["robust_pr_auc"].agg(new_mean="mean", new_std="std", new_n="count")

cc = old_g.join(fresh_g)
cc["delta"] = cc["new_mean"] - cc["old_mean"]
# Beyond seed noise = |delta| exceeds the summed 1-sigma envelopes (floor 0.01).
noise = (cc["old_std"].fillna(0) + cc["new_std"].fillna(0)).clip(lower=0.01)
cc["flag"] = np.where(cc["delta"].abs() > noise, "DEVIATES", "ok")
cc = cc.round(4).reset_index()
CONSISTENCY_CSV = os.path.join(OUT_DIR, "thesis_consistency_check.csv")
cc.to_csv(CONSISTENCY_CSV, index=False)
print(f"Saved -> {CONSISTENCY_CSV}\n")
print(cc.to_string(index=False))

# CCFD variance reconciliation (spec §6.5): thesis 0.581±0.102 vs ICDM draft 0.580±0.225.
ccfd_new = fresh_a[fresh_a["dataset"] == "CCFD"]["robust_pr_auc"]
ccfd_old = old_a[old_a["dataset"] == "CCFD"]["robust_pr_auc"]
print(f"\nCCFD reconciliation: old baseline {ccfd_old.mean():.3f}±{ccfd_old.std():.3f} "
      f"(thesis 0.581±0.102) | fresh {ccfd_new.mean():.3f}±{ccfd_new.std():.3f} "
      f"(n={len(ccfd_new)}) | the draft's ±0.225 matches neither -> use the fresh value.")

In [ ]:
# Cell 7: Golden-reference anchors (spec §4.4 / §6.6) — warn-only self-check
# Anchors encode the spec-§4.4 table (means over 3 seeds, no-defence MLP,
# CAPGD eps=0.1) plus the §4.4 cross-checks. C2 anchors came from BETWEEN-model
# thesis runs; the regenerated grid is same-model, so C2 deviations are
# recorded, never force-fitted (spec §4.4 caveat).
ANCHORS = [
    # (dataset, protocol, metric, anchor_value, tolerance)
    ("LCLD", "A_unconstrained",    "robust_pr_auc",          0.105,  0.02),
    ("LCLD", "A_unconstrained",    "feasible_flipped_count", 3,      3),
    ("LCLD", "B_posthoc_filter",   "robust_pr_auc",          0.30,   0.03),
    ("LCLD", "C1_projection",      "robust_pr_auc",          0.105,  0.02),
    ("LCLD", "C1_projection",      "feasible_flipped_count", 2111,   250),
    ("LCLD", "C2_mask_projection", "robust_accuracy",        0.153,  0.05),
    ("LCLD", "C2_mask_projection", "fsr",                    1.00,   0.001),
    ("IEEE-CIS", "A_unconstrained",    "robust_pr_auc",          0.068,  0.02),
    ("IEEE-CIS", "A_unconstrained",    "feasible_flipped_count", 0,      1),
    ("IEEE-CIS", "B_posthoc_filter",   "robust_pr_auc",          0.43,   0.03),
    ("IEEE-CIS", "C1_projection",      "robust_pr_auc",          0.065,  0.02),
    ("IEEE-CIS", "C1_projection",      "feasible_flipped_count", 81,     60),
    ("IEEE-CIS", "C2_mask_projection", "robust_pr_auc",          0.409,  0.05),
    ("IEEE-CIS", "C2_mask_projection", "robust_accuracy",        0.883,  0.05),
    ("IEEE-CIS", "C2_mask_projection", "feasible_flipped_count", 7,      6),
    ("CCFD",    "A_unconstrained", "robust_pr_auc",         0.58,   0.08),
    ("Sparkov", "A_unconstrained", "robust_pr_auc",         0.005,  0.01),
    ("Sparkov", "A_unconstrained", "aggregate_feasibility", 0.0001, 0.0005),
]
core = summ[(summ["attack"] == "CAPGD") & (summ["defence"] == "none") & (summ["epsilon"] == 0.1)]
g_rows = []
for ds, proto, metric, val, tol in ANCHORS:
    got = core[(core["dataset"] == ds) & (core["protocol"] == proto)][f"mean_{metric}"]
    got = float(got.iloc[0]) if len(got) else float("nan")
    status = "ok" if abs(got - val) <= tol else "WARN"
    g_rows.append({"dataset": ds, "protocol": proto, "metric": metric,
                   "anchor_value": val, "tolerance": tol,
                   "fresh_value": round(got, 4), "status": status})
gold = pd.DataFrame(g_rows)
GOLD_CSV = os.path.join(OUT_DIR, "golden_reference_anchors.csv")
gold.to_csv(GOLD_CSV, index=False)
n_warn = int((gold["status"] == "WARN").sum())
print(f"Saved {len(gold)} anchors -> {GOLD_CSV}  ({n_warn} WARN)\n")
print(gold.to_string(index=False))

In [ ]:
# Cell 8: fig_kendall_tau (+ free variant) — rank bump charts (spec §6.7)
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams.update({"font.family": "serif", "font.size": 9})

def ranks(scores):
    """Rank positions (1 = most robust) for a score list."""
    order = np.argsort(-np.asarray(scores))
    r = np.empty(len(scores), dtype=int)
    r[order] = np.arange(1, len(scores) + 1)
    return r

def bump_panel(ax, items, s_left, s_right, label_left, label_right, title):
    rl, rr = ranks(s_left), ranks(s_right)
    for i, item in enumerate(items):
        ax.plot([0, 1], [rl[i], rr[i]], marker="o", lw=1.5)
        ax.text(-0.08, rl[i], item, ha="right", va="center", fontsize=7)
        ax.text(1.08, rr[i], item, ha="left", va="center", fontsize=7)
    ax.set_xlim(-1.1, 2.1)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([label_left, label_right])
    ax.set_ylim(len(items) + 0.5, 0.5)
    ax.set_yticks(range(1, len(items) + 1))
    ax.set_title(title, fontsize=8)

# Strong: defences under B -> C1 (row 0) and B -> C2 (row 1), per dataset.
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for j, ds in enumerate(["IEEE-CIS", "LCLD", "Sparkov"]):
    sub = capgd[capgd["dataset"] == ds]
    by = {p: [sub[(sub["defence"] == d) & (sub["protocol"] == p)]["robust_pr_auc"].mean()
              for d in DEFENCES]
          for p in ["B_posthoc_filter", "C1_projection", "C2_mask_projection"]}
    for i, (c_proto, c_lab) in enumerate([("C1_projection", "C1"), ("C2_mask_projection", "C2")]):
        d = kt[(kt["comparison"] == "strong_defence_B_vs_C") & (kt["dataset"] == ds)
               & (kt["protocol_right"] == c_proto)]["kt_distance"].iloc[0]
        bump_panel(axes[i, j], DEFENCES, by["B_posthoc_filter"], by[c_proto],
                   "B", c_lab, f"{ds}  B vs {c_lab}  (KT dist {d:.2f})")
fig.suptitle("Defence-ranking stability across protocols (CAPGD MLP, ε=0.1, weighted KT)",
             fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.94])
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(FIG_DIR, f"fig_kendall_tau.{ext}"), dpi=200, bbox_inches="tight")
plt.show()

# Free: model families under A -> B (Square), per dataset.
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for j, ds in enumerate(["CCFD", "IEEE-CIS", "LCLD", "Sparkov"]):
    sub = square[square["dataset"] == ds]
    by = {p: [sub[(sub["model"] == m) & (sub["protocol"] == p)]["robust_pr_auc"].mean()
              for m in MODELS]
          for p in ["A_unconstrained", "B_posthoc_filter"]}
    d = kt[(kt["comparison"] == "free_model_A_vs_B") & (kt["dataset"] == ds)]["kt_distance"].iloc[0]
    bump_panel(axes[j], MODELS, by["A_unconstrained"], by["B_posthoc_filter"],
               "A", "B", f"{ds}  (KT dist {d:.2f})")
fig.suptitle("Model-family ranking stability: Protocol A vs B (Square, ε=0.1)", fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.92])
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(FIG_DIR, f"fig_kendall_tau_free.{ext}"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved fig_kendall_tau.{pdf,png} + fig_kendall_tau_free.{pdf,png}")

In [ ]:
# Cell 9: experiment_status.md (spec §8 template, filled with computed findings)
ieee_c2_acc = float(core[(core["dataset"] == "IEEE-CIS")
                         & (core["protocol"] == "C2_mask_projection")]["mean_robust_accuracy"].iloc[0])
lcld_c2_acc = float(core[(core["dataset"] == "LCLD")
                         & (core["protocol"] == "C2_mask_projection")]["mean_robust_accuracy"].iloc[0])
spark_agg = float(core[(core["dataset"] == "Sparkov")
                       & (core["protocol"] == "A_unconstrained")]["mean_aggregate_feasibility"].iloc[0])
spark_bind = fresh_a[fresh_a["dataset"] == "Sparkov"]["main_failed_constraint"].mode().iloc[0]
ccfd_mlp_clean_nb2 = float(nb2[(nb2["model"] == "MLP") & (nb2["protocol"] == "A_unconstrained")
                               & (nb2["dataset"] == "CCFD")]["clean_pr_auc"].mean())
strong = kt[kt["comparison"] == "strong_defence_B_vs_C"]
kt_lines = "\n".join(
    f"  {r.dataset}: B-vs-{r.protocol_right[:2]} weighted-KT distance = {r.kt_distance}"
    f"  ({r.ranking_left}  ->  {r.ranking_right})"
    for r in strong.itertuples())

status = f"""Decisions: HopSkipJump dropped from ICDM scope (future work); all results regenerated fresh.

Completed:
- NB1 CAPGD protocol grid: MLP x 4 datasets x {{none,AT,input_validation}} x 3 seeds,
  protocols A/B/C1/C2 (CCFD A/B only), eps sweep on no-defence; folded-OHE aggregate.
  ({len(nb1)} rows, single weight hash per same_model_group.)
- NB2 Square model-family: 4 datasets x 3 models x {{A,B}}, adversarial examples saved.
  ({len(nb2)} rows incl. 24 protocol=not_applicable Protocol-C placeholders.)
- NB3 consolidation: master registry ({len(master)} rows), coverage + summary,
  Kendall-Tau (strong + free), PR-AUC vs ROC-AUC, thesis-consistency cross-check,
  golden anchors ({n_warn} WARN), figures.

Findings to confirm in writing:
- Same-model C2 vs between-model thesis Tables 11-12: IEEE-CIS robust acc
  {ieee_c2_acc:.3f} (anchor 0.883 - matches); LCLD robust acc {lcld_c2_acc:.3f}
  (anchor 0.153 - small same-model shift, recorded not force-fitted).
- Sparkov folded aggregate: binding constraint {spark_bind} (expected s_state_ohe);
  aggregate {spark_agg:.6f} (~0.0001 as expected).
- CCFD robust PR-AUC reconciled to: {cc[cc['dataset'] == 'CCFD']['new_mean'].iloc[0]:.3f}
  +/- {cc[cc['dataset'] == 'CCFD']['new_std'].iloc[0]:.3f} (vs thesis 0.581+/-0.102;
  the ICDM draft's +/-0.225 matches neither registry and is superseded).
  NB2 note: CCFD MLP clean PR-AUC {ccfd_mlp_clean_nb2:.3f} vs anchor 0.633 - within
  CCFD's known seed variance.
- Strong B-vs-C Kendall-Tau distances per dataset:
{kt_lines}

Deferred (future work):
- HopSkipJump (6 cached rows exist, unused); black-box in-attack Protocol C for
  tree/ensemble (Cartella-style); CAA/MOEVA head-to-head; FA-AT; CTGAN;
  CCFD extra seeds; M2 ordinal.
"""
STATUS_MD = os.path.join(OUT_DIR, "experiment_status.md")
with open(STATUS_MD, "w") as f:
    f.write(status)
print(f"Saved -> {STATUS_MD}\n")
print(status)

## Notes

- **Single provenance:** every number in `icdm_master_registry.csv` comes from
  the regenerated NB1/NB2 runs (same-model, hash-verified). The old
  `registry_clean.csv` is touched exactly once, read-only, in Cell 6.
- **Placeholder filter:** all metric computations exclude
  `protocol == "not_applicable"` (NB2's pandas-safe encoding of spec §5.2's
  `N/A` marker).
- **Old-registry selection (Cell 6):** the `*baseline` experiments are the
  canonical comparison; the `*eps_sweep` rows duplicate the same config at
  eps=0.1 and would artificially shrink the variance being reconciled.
- **KT with 3-item rankings** is coarse (distances quantised to
  {0, 1/3, 2/3, 1} for plain tau); the weighted variant emphasises the
  top-ranked defence/model. NaN appears if a score vector is constant.
- **C2 anchors** are between-model thesis values; the same-model regeneration
  may legitimately shift them (spec §4.4 caveat) — Cell 7 warns, never fails.
